# Desktop Helper — Fine-tuning on Colab

Fine-tunes `DesktopHelperLM` (OPT-350M weights transplanted into the custom architecture) on Dolly + synthetic tool-call data.

**Prerequisites in Google Drive:**
- `opt_transplant.pt` — the transplanted checkpoint (produced locally by `model/load_opt.py`)

**Runtime:** set to GPU (Runtime → Change runtime type → T4 GPU or better).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone the repo (pretrained-model branch)

In [ ]:
!git clone https://github.com/JaysonSalemmo/Desktop_Helper.git
%cd Desktop_Helper
!git checkout pretrained-model

## 3. Install dependencies

Colab ships with torch + CUDA. We only need the data/tokenizer libraries.

In [ ]:
!pip install -q datasets tokenizers

## 4. Regenerate the tool-call data

`data/tool_calls.jsonl` is gitignored, so regenerate it here (deterministic — same seed as local).

In [ ]:
!python -m model.data.tool_calls --count 500 --seed 42 --output data/tool_calls.jsonl

## 5. Confirm GPU is available

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU'
print(torch.cuda.get_device_name(0))

## 6. Fine-tune

Adjust `--checkpoint` to wherever you uploaded `opt_transplant.pt` in Drive.
Checkpoints save to Drive after every epoch, so they survive session resets.

If you hit CUDA out-of-memory on a T4, lower `--batch-size` to 2 and raise `--grad-accum` to 16 (keeps the effective batch at 32).

In [ ]:
!python -m model.train \
  --checkpoint /content/drive/MyDrive/opt_transplant.pt \
  --tokenizer model/tokenizer.json \
  --tool-calls data/tool_calls.jsonl \
  --output /content/drive/MyDrive/desktop_helper_checkpoints/ \
  --epochs 3 \
  --batch-size 4 \
  --grad-accum 8